In [11]:
import sys
print(sys.executable)

D:\Programming\Python\AI\F1 Fantasy\.venv\Scripts\python.exe


In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBRegressor

In [13]:
df = pd.read_csv('train.csv', index_col='id')

X = df.drop(columns=['PitNextLap', 'PitStop', 'RaceProgress'], axis=1)
y = df['PitNextLap']

categories = ["Driver", "Compound", "Race"]
X = pd.get_dummies(X, columns=categories)

In [14]:
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

prediction = model.predict(X_test)
print(prediction)

[0 0 0 ... 1 0 0]


In [6]:
print(type(model))

pred = model.predict(X_test)

print(pred[:10])
print(y_test[:10])

<class 'xgboost.sklearn.XGBClassifier'>
[0 0 0 1 0 1 0 0 1 0]
id
287071    0.0
317834    0.0
435371    0.0
350788    1.0
68911     0.0
259781    1.0
99784     0.0
408396    0.0
151263    0.0
163334    0.0
Name: PitNextLap, dtype: float64


In [7]:
print(accuracy_score(y_test, prediction))

0.8940542879264016


In [10]:
decay = 0.85
list = [0.6, 0.55, 0.5, 0.45, 0.20, 0.15]

sum = 0
decaySum = 0
decayWeights = 0
for i in range(len(list)):
    sum += list[i]
    if i != 0:
        decaySum += list[i] * (decay ** i)
        decayWeights += decay ** i
    else:
        decaySum += list[i]
        decayWeights += 1

print(sum/len(list))
print(decaySum/decayWeights)
print(decaySum)
print(decayWeights)

0.4083333333333334
0.45180906427909534
1.876063296875
4.1523365624999995


In [1]:
import fastf1
import pandas as pd
fastf1.Cache.enable_cache('D:\\Programming\\Python\\AI\\F1 Fantasy\\cache')

In [2]:
events = fastf1.get_events_remaining()
nextRound = events.iloc[0].RoundNumber

In [4]:
qpace = {}

# session = fastf1.get_session(2026, nextRound-1, 'Q')
# session.load(telemetry=False)
# for driver in session.drivers:
#     qpace[driver] = []

for round in range(1, nextRound):
    session = fastf1.get_session(2026, round, 'Q')
    session.load(telemetry=False)

    qualy_laps = session.results[['FullName', 'Position', 'Q1', 'Q2', 'Q3']]

    fastestLap = qualy_laps.iloc[0].Q3.total_seconds()

    for index, driver in qualy_laps.iterrows():
        lapTime = 0
        if pd.isna(driver.Q3):
            if pd.isna(driver.Q2):
                lapTime = driver.Q1.total_seconds()
            else:
                lapTime = driver.Q2.total_seconds()
        else:
            lapTime = driver.Q3.total_seconds()

        delta = lapTime - fastestLap
        percent = delta/fastestLap*100

        if not str(index) in qpace:
            qpace[str(index)] = []
        qpace[str(index)].append(percent)

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 18
core        WARNING 	No lap data for driver 55
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 55)
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '6', '16', '81', '1',

In [5]:
qpace

{'63': [0.0,
  0.2411366006256609,
  0.3356687467615711,
  0.45445226542745953,
  0.0,
  0.5468348808482759,
  0.0,
  0.0,
  0.41992486749666935,
  0.48677187838368313,
  0.716256298004082,
  0.14333291176595123],
 '12': [0.37316284164141517,
  0.0,
  0.0,
  0.0,
  0.09369230345283397,
  0.0,
  0.4271615849167807,
  0.45528110961535845,
  0.0,
  0.0,
  0.3522996619477581,
  0.1868948751458058],
 '6': [0.9997707532030828,
  1.148114355231146,
  1.3516862285701283,
  nan,
  0.491884593127393,
  0.5315679171697764,
  0.5329476827488265,
  0.7850195876756545,
  0.7206818671902383,
  1.4009064688916253,
  0.8405973551620979],
 '16': [1.0303369927914587,
  0.395377128953776,
  0.7062560544278935,
  0.39294744755005684,
  0.5483755407974815,
  0.41637173668650973,
  0.8061168467708509,
  0.35696459092766053,
  0.19861311300518342,
  0.5097689750002361,
  0.3082622042042814,
  0.5550637269367652],
 '81': [1.0978374385491159,
  0.5278936392075124,
  0.3987474374281907,
  0.7995626324062031,
  0

In [6]:
decay = 0.85
predictedQPace = {}

for driverNumber in qpace:
    decaySum = 0
    decayWeights = 0
    
    for i in range(len(qpace[driverNumber])):
        # get opposite index to iterate backwards (to go by most recent values)
        j = len(qpace[driverNumber]) - i - 1
        if not pd.isna(qpace[driverNumber][j]):
            if i != 0:
                decaySum += (qpace[driverNumber][j]) * (decay ** i) 
                decayWeights += decay ** i
            else:
                decaySum += qpace[driverNumber][j]
                decayWeights += 1

    predictedQPace[driverNumber] = decaySum/decayWeights
    #print(f"{driverNumber}: {decaySum/decayWeights}")

predictedQPace

{'63': 0.31775079921190563,
 '12': 0.17592805401767597,
 '6': 0.8727345174670739,
 '16': 0.4730581124949286,
 '81': 0.588626855166067,
 '1': 0.41651847882702253,
 '44': 0.3944260596169346,
 '30': 1.6741805679982054,
 '41': 1.5605927005255962,
 '5': 2.0901324764836198,
 '27': 2.1655156841519703,
 '87': 2.902518475378515,
 '31': 2.9104631800824246,
 '10': 1.8988605330772188,
 '23': 3.2523311492057654,
 '43': 2.383051197200566,
 '14': 4.457917159472234,
 '11': 4.352871685925667,
 '77': 4.271535310467523,
 '18': 4.920090515335197,
 '3': 0.5891313293791217,
 '55': 3.138293817630431,
 '22': 2.057248851228867}

In [24]:
session = fastf1.get_session(2026, nextRound-1, 'FP3')
session.load(telemetry=False)
# Unlike for qualy, session.results doesn't give the lap time, so lap time is manually gotten from session.laps
results = (
    session.laps.pick_quicklaps()
    .groupby('DriverNumber')[['Driver', 'LapTime']]
    .min(numeric_only=False)
    .sort_values(by='LapTime')
)

results

core           INFO 	Loading data for Hungarian Grand Prix - Practice 3 [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 11
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 11)
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30', '31', '41', '43', '44', '55', '63', '77', '81', '87']


,Driver,LapTime
DriverNumber,,
1,NOR,0 days 00:01:17.939000
44,HAM,0 days 00:01:18.056000
12,ANT,0 days 00:01:18.068000
16,LEC,0 days 00:01:18.291000
81,PIA,0 days 00:01:18.438000
63,RUS,0 days 00:01:18.541000
3,VER,0 days 00:01:18.656000
6,HAD,0 days 00:01:18.943000
30,LAW,0 days 00:01:19.088000


In [7]:
practicePace = {}

for i in range(1, 4):
    try:
        session = fastf1.get_session(2026, nextRound, f'FP{i}')
        session.load(telemetry=False)
    except Exception as e:
        print(e)
        continue

    results = (
        session.laps.pick_quicklaps()
        .groupby('DriverNumber')[['Driver', 'LapTime']]
        .min(numeric_only=False)
        .sort_values(by='LapTime')
    )

    fastestLap = results.iloc[0].LapTime.total_seconds()

    for index, driver in results.iterrows():
        lapTime = driver.LapTime.total_seconds()

        delta = lapTime - fastestLap
        percent = delta/fastestLap*100

        if not str(index) in practicePace:
            practicePace[str(index)] = []
        practicePace[str(index)].append(percent)

practicePace

core           INFO 	Loading data for Italian Grand Prix - Practice 1 [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '5', '12', '14', '16', '18', '22', '25', '27', '30', '31', '36', '41', '43', '44', '46', '55', '61', '63', '77', '81', '87']
core           INFO 	Loading data for Italian Grand Prix - Practice 2 [v3.8.3]
req         

{'16': [0.0, 0.14535059775433878, 0.5947530376190472],
 '44': [0.20841364687741162, 0.5535435264477621, 0.2748756370182064],
 '63': [0.3662297609868953, 0.0, 0.0],
 '30': [0.511998843484979, 1.3335917343960066, 1.1968036585217638],
 '12': [0.7661912104857482, 0.17078695236134805, 0.43907126090077014],
 '1': [0.8565439475713166, 0.4651219128138669, 0.4938031355282914],
 '41': [0.9565343099460428, 0.9568914352160349, 0.6142132597088382],
 '5': [1.2022937548188184, 1.474097312225195, 1.3123487271798584],
 '43': [1.2287972243639291, 1.2839302801632801, 1.0727447426993821],
 '61': [1.4082979953739534],
 '81': [1.4167309175019298, 0.568078586223196, 0.6738101898587943],
 '22': [1.8829510408635342, 1.085284463232356, 1.9435896812172357],
 '27': [1.9492097147263028, 1.42080209304861, 1.1031513397146775],
 '87': [1.9733037779491198, 0.9823277898230441, 1.6614164609153599],
 '46': [2.0865458750963755],
 '55': [2.191355050115655, 1.6242929299046842, 2.0141329862927138],
 '36': [2.2467713955281527

In [8]:
decay = 0.6

predictedPracticePace = {}

for driverNumber in practicePace:
    decaySum = 0
    decayWeights = 0
    
    for i in range(len(practicePace[driverNumber])):
        # get opposite index to iterate backwards (to go by most recent values)
        j = len(practicePace[driverNumber]) - i - 1
        if not pd.isna(practicePace[driverNumber][j]):
            if i != 0:
                decaySum += (practicePace[driverNumber][j]) * (decay ** i) 
                decayWeights += decay ** i
            else:
                decaySum += practicePace[driverNumber][j]
                decayWeights += 1

    predictedPracticePace[driverNumber] = decaySum/decayWeights
    #print(f"{driverNumber}: {decaySum/decayWeights}")

predictedPracticePace

{'16': 0.3479405083018625,
 '44': 0.34797482947078157,
 '63': 0.06726669079351137,
 '30': 1.1128970830683471,
 '12': 0.41702666739410627,
 '1': 0.5516490328276966,
 '41': 0.7819900369484871,
 '5': 1.341649421555995,
 '43': 1.1660560773307984,
 '61': 1.4082979953739534,
 '81': 0.7778981999456156,
 '22': 1.6697054764630213,
 '27': 1.3557898432884248,
 '87': 1.5108175994239132,
 '46': 2.0865458750963755,
 '55': 1.9273451848352856,
 '36': 2.2467713955281527,
 '14': 3.203947057400793,
 '31': 2.4321194167010827,
 '77': 3.3470693505825952,
 '18': 3.745695139544892,
 '3': 0.6376101893932937,
 '10': 1.067575814565024,
 '23': 2.27380728416683,
 '11': 3.283582389344105}

In [10]:
predictedQPace['16'] * 0.4 + predictedPracticePace['16'] * 0.6

prediction = {}

for driver in predictedQPace:
    if driver in predictedPracticePace:
        prediction[driver] = predictedQPace[driver] * 0.35 + predictedPracticePace[driver] * 0.65
        #print(driver, ":", predictedQPace[driver] * 0.35 + predictedPracticePace[driver] * 0.65)

sortedData = dict(sorted(prediction.items(), key=lambda item: item[1]))
sortedData

{'63': 0.15493612873994936,
 '12': 0.3326421527123557,
 '44': 0.36423276002193516,
 '16': 0.39173166976943563,
 '1': 0.5043533389274607,
 '3': 0.6206425883883335,
 '81': 0.7116532292727735,
 '41': 1.0545009692004752,
 '30': 1.3093463027937975,
 '10': 1.3585254660442923,
 '43': 1.592004369285217,
 '5': 1.6036184907806637,
 '27': 1.6391938875906655,
 '22': 1.8053456576310674,
 '87': 1.9979129060080238,
 '55': 2.3511772063135865,
 '31': 2.5995397338845523,
 '23': 2.6162906369304575,
 '14': 3.6428365931257973,
 '11': 3.6578336431476517,
 '77': 3.67063243654232,
 '18': 4.156733521071499}

In [22]:
import numpy as np
import pandas as pd

def build_rpace_params(
    qpace_df: pd.DataFrame,
    grid_col: str = "grid_pos",
    penalty_col: str = "grid_penalty",
    qpace_col: str = "qpace",
    driver_col: str = "Driver",
    team_col: str = "Team",
    # model knobs (good starting defaults)
    k_overtake: float = 0.045,          # sec/lap gain per expected overtake
    k_penalty: float = 0.030,           # sec/lap loss per penalty place * track_difficulty
    track_difficulty: float = 1.0,      # 0.7 easy overtaking, 1.3 hard overtaking
    base_sd: float = 0.22,              # baseline race-pace uncertainty
    sd_scale_qgap: float = 0.12,        # more uncertainty farther from front
    sd_penalty_boost: float = 0.01      # uncertainty added per penalty place
) -> pd.DataFrame:
    """
    Input columns expected in qpace_df:
      - Driver (or driver_col)
      - qpace (seconds/lap; lower is better)
      - grid_pos (1..20)
      - grid_penalty (>=0 integer, optional but recommended)
      - Team (optional)
    Returns:
      driver-level parameters for simulation:
      - rpace_mean, rpace_sd
    """
    df = qpace_df.copy()

    # required columns
    required = {driver_col, qpace_col, grid_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if penalty_col not in df.columns:
        df[penalty_col] = 0

    # normalize types
    df[qpace_col] = pd.to_numeric(df[qpace_col], errors="coerce")
    df[grid_col] = pd.to_numeric(df[grid_col], errors="coerce")
    df[penalty_col] = pd.to_numeric(df[penalty_col], errors="coerce").fillna(0)

    df = df.dropna(subset=[qpace_col, grid_col]).copy()

    # 1) baseline from Qpace
    # shrink very slightly to field median to avoid overfitting one session
    field_med = df[qpace_col].median()
    df["rpace_base"] = 0.90 * df[qpace_col] + 0.10 * field_med

    # 2) expected overtakes from pace-vs-grid mismatch
    # convert qpace rank to "fair" starting expectation
    df["q_rank"] = df[qpace_col].rank(method="average", ascending=True)  # 1 fastest
    # positive -> expected to gain places, negative -> expected to lose
    df["expected_overtakes"] = df[grid_col] - df["q_rank"]

    # clip extreme values for stability
    df["expected_overtakes"] = df["expected_overtakes"].clip(-8, 8)

    # 3) combine to race pace mean
    # lower rpace is better, so overtakes reduce time, penalties increase time
    df["rpace_mean"] = (
        df["rpace_base"]
        - k_overtake * df["expected_overtakes"]
        + k_penalty * df[penalty_col] * track_difficulty
    )

    # 4) uncertainty model (for MC sampling)
    # extra uncertainty for drivers farther from front + with penalties
    qgap = (df[qpace_col] - df[qpace_col].min()).clip(lower=0)
    df["rpace_sd"] = (
        base_sd
        + sd_scale_qgap * qgap
        + sd_penalty_boost * df[penalty_col]
    )

    # guardrails
    df["rpace_sd"] = df["rpace_sd"].clip(lower=0.12, upper=0.65)

    # optional convenience: event-relative delta
    event_med_rpace = df["rpace_mean"].median()
    df["rpace_delta"] = df["rpace_mean"] - event_med_rpace

    out_cols = [driver_col]
    if team_col in df.columns:
        out_cols.append(team_col)
    out_cols += [
        qpace_col, grid_col, penalty_col,
        "expected_overtakes", "rpace_mean", "rpace_sd", "rpace_delta"
    ]

    return df[out_cols].sort_values("rpace_mean").reset_index(drop=True)

In [23]:
# Example input columns:
# Driver, Team, qpace, grid_pos, grid_penalty
rpace_params = build_rpace_params(
    qpace_df=predictedQPace,
    track_difficulty=1.15,   # e.g. harder overtaking track
    k_overtake=0.045,
    k_penalty=0.030
)
rpace_params.head(20)

AttributeError: 'dict' object has no attribute 'columns'